In [ ]:
import torch
import torch.nn.functional as F
import math

def span_refreshed_attention(x, Wq, Wk, Wv, Wq_in, Wk_in, Wv_in,
                             include_query_token=True):
    """
    Single head, no batch, for clarity.
    x: (L, d) hidden states.
    Wq, Wk, Wv:        (d, d) OUTER projections.
    Wq_in, Wk_in, Wv_in:(d, d) INNER projections (used inside each span).

    For outer query j and key i <= j:
      token i runs an inner attention with its inner query over the span
      [i .. j], yielding a refreshed repr u_{i,j}; the outer key/value that
      i presents to j are derived from u_{i,j}. Outer attention is then the
      usual softmax for j over i <= j.
    Returns: (L, d).
    """
    L, d = x.shape
    scale = 1.0 / math.sqrt(d)

    Qo = x @ Wq                 # (L, d) outer queries, one per token
    Qi = x @ Wq_in              # (L, d) inner queries  (token i attends span)
    Ki = x @ Wk_in              # (L, d) inner keys
    Vi = x @ Wv_in              # (L, d) inner values

    out = torch.zeros(L, d)

    for j in range(L):
        k_ij = torch.zeros(j + 1, d)   # refreshed keys i presents to j
        v_ij = torch.zeros(j + 1, d)   # refreshed values i presents to j

        for i in range(j + 1):
            hi = j if include_query_token else max(i, j - 1)
            span = slice(i, hi + 1)            # tokens [i .. hi], capped at j

            a = (Qi[i] @ Ki[span].T) * scale   # (span_len,)
            b = F.softmax(a, dim=-1)           # inner weights
            u = b @ Vi[span]                   # (d,) refreshed repr of i wrt j

            k_ij[i] = u @ Wk                   # outer key from refreshed repr
            v_ij[i] = u @ Wv                   # outer value from refreshed repr

        s = (Qo[j] @ k_ij.T) * scale           # (j+1,) outer scores
        p = F.softmax(s, dim=-1)               # outer weights
        out[j] = p @ v_ij

    return out